In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

dicom_tags = pd.read_csv(r'output\dicomtags.csv')
view_ident_avi = pd.read_csv(r'output\viewpredictions.csv')
view_ident_img = pd.read_csv(r'output\viewpredictions_img.csv')


In [ ]:
small_pct = 5.0


def autopct_big_only(values, min_pct=5.0):
    total = np.sum(values)

    def inner(pct):
        n = int(round(pct * total / 100.0))
        if pct < min_pct:
            return ''
        return f'{pct:.1f}%\n({n:,})'

    return inner


def donut_plot_with_outer_labels(
    series, title, ax=None, cmap=plt.cm.YlGnBu, small_pct=5.0,
    fontsize=16, title_fontsize=16
):
    counts = series.fillna('N/A').astype(str).value_counts(dropna=False)
    labels = counts.index.to_list()
    values = counts.to_numpy()
    total = values.sum()

    if total == 0:
        raise ValueError(f'No hay datos para representar en {title!r}.')

    colors = cmap(np.linspace(0.25, 0.95, len(values)))

    if ax is None:
        _, ax = plt.subplots(figsize=(8, 8))

    wedges, _, _ = ax.pie(
        values,
        labels=None,
        colors=colors,
        startangle=90,
        counterclock=False,
        autopct=autopct_big_only(values, min_pct=small_pct),
        pctdistance=0.78,
        wedgeprops=dict(width=0.42, edgecolor='white'),
        textprops=dict(fontsize=fontsize, ha='center', va='center'),
    )

    small_right = []
    small_left = []

    for i, wedge in enumerate(wedges):
        angle = (wedge.theta1 + wedge.theta2) / 2
        x = np.cos(np.deg2rad(angle))
        y = np.sin(np.deg2rad(angle))
        pct = 100 * values[i] / total

        if pct < small_pct:
            entry = dict(
                x=x, y=y, pct=pct, n=values[i], label=labels[i]
            )
            (small_right if x >= 0 else small_left).append(entry)
        else:
            ax.text(
                1.06 * x,
                1.06 * y,
                labels[i],
                ha='left' if x >= 0 else 'right',
                va='center',
                fontsize=fontsize,
            )

    def place_small_group(items, side):
        if not items:
            return

        items = sorted(items, key=lambda item: item['y'], reverse=True)
        y_targets = np.linspace(1.18, 0.92, len(items))
        x_text = 1.42 if side == 'right' else -1.42
        ha = 'left' if side == 'right' else 'right'

        for item, y_text in zip(items, y_targets):
            text = (
                f"{item['label']}  {item['pct']:.1f}% ({item['n']:,})"
            )
            ax.annotate(
                text,
                xy=(0.99 * item['x'], 0.99 * item['y']),
                xytext=(x_text, y_text),
                ha=ha,
                va='center',
                fontsize=fontsize,
                arrowprops=dict(
                    arrowstyle='-',
                    lw=0.9,
                    color='0.35',
                    connectionstyle='angle3,angleA=0,angleB=90',
                ),
            )

    place_small_group(small_right, 'right')
    place_small_group(small_left, 'left')

    ax.text(
        0, 0, f'N = {total:,}',
        ha='center', va='center',
        fontsize=fontsize, fontweight='bold',
    )

    ax.set_title(title, fontsize=title_fontsize)
    ax.set_aspect('equal')
    return ax


In [ ]:
# 1. Modality stored directly in the DICOM tags file
fig, ax = plt.subplots(figsize=(8, 8))
modality_fig = fig
donut_plot_with_outer_labels(
    dicom_tags['Modality'],
    'Modality',
    ax=ax,
    small_pct=small_pct,
)
plt.tight_layout()
plt.show()


In [ ]:
# 2. Predicted views for videos
view_acronyms = {
    'Doppler_Parasternal_Short': 'PSAX_DP',
    'Doppler_Parasternal_Long': 'PLAX_DP',
    'Apical_Doppler': 'AP_DP',
    'Parasternal_Long': 'PLAX',
    'Parasternal_Short': 'PSAX',
}

fig, ax = plt.subplots(figsize=(8, 8))
avi_views_fig = fig
donut_plot_with_outer_labels(
    view_ident_avi['predicted_view'].replace(view_acronyms),
    'Views — videos',
    ax=ax,
    small_pct=small_pct,
    fontsize=11,
    title_fontsize=14,
)
plt.tight_layout()
plt.show()


In [ ]:
# 3. View breakdown for single-frame B-mode images
fig, ax = plt.subplots(figsize=(8, 8))
single_frame_views_fig = fig
donut_plot_with_outer_labels(
    view_ident_img['predicted_view'].replace(view_acronyms),
    'Views — B-mode single-frame',
    ax=ax,
    small_pct=small_pct,
    fontsize=11,
    title_fontsize=14,
)
plt.tight_layout()
plt.show()
